# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains mass spectrometry analysis of protein abundance, modifications, and sequences in human extracellular vesicles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print("Published:", getattr(metadata, 'datePublished', 'N/A'))
print("Authors: (first 3 if available)")
authors = getattr(metadata, 'author', [])
for author in (authors if isinstance(authors, list) else [authors])[:3]:
    print(" -", author.get('@id', author))

## 2. Data Overview
Review available record sets, their `@id`s, and the associated fields in the dataset schema.

In [ ]:
# List available record sets in the dataset (by @id)
record_sets = [rs for rs in getattr(metadata, 'recordSet', [])]
if not record_sets:
    print("No record sets found in dataset metadata.\nPerhaps you should check for attached files or documentation in 'distribution'.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '')}")
        # List fields for each record set (by @id)
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields by @id:")
        for field in fields:
            print(f"    - {field['@id']} ({field.get('name', '')})")
        print()

# If the above block prints no record sets, print distribution files as an alternative (because this dataset attaches data files in 'distribution').
if not record_sets:
    print("Available distribution(s):")
    dists = getattr(metadata, 'distribution', [])
    for dist in dists:
        print("  @id:", dist.get('@id', dist))

## 3. Data Extraction
In the absence of explicit RecordSets in the schema, we'll enumerate the CSV/TSV files listed in the `distribution` section and use the distribution `@id`s. We'll attempt to load them as record sets using their IDs.

Loading data into a DataFrame for analysis.

In [ ]:
# Load tabular data files as record sets by their distribution @id
dataframes = {}
tabular_distributions = []
for dist in getattr(metadata, 'distribution', []):
    # For mlcroissant, the distribution @id can be used for record extraction
    # We'll check if it's a CSV or TSV
    url = dist.get('contentUrl', '')
    enc_format = dist.get('encodingFormat', '')
    # Heuristically treat as tabular if it contains csv/tsv in 'encodingFormat' or file extension
    if any(s in (enc_format or '').lower() for s in ['csv', 'tsv']) or url.endswith('.csv') or url.endswith('.tsv'):
        tabular_distributions.append(dist)

print(f"Found {len(tabular_distributions)} potential tabular files in distribution.")

for dist in tabular_distributions:
    rec_set_id = dist['@id']
    print(f"Loading records from distribution @id: {rec_set_id}")
    try:
        records = list(dataset.records(record_set=rec_set_id))
        df = pd.DataFrame(records)
        dataframes[rec_set_id] = df
        print(f"  - Dataframe shape: {df.shape}")
        print(f"  - Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"  - Error loading records from {rec_set_id}: {e}")

if not dataframes:
    print('Could not extract any tabular data from distributions.')
else:
    # Let's pick the first one as an example
    primary_recset_id = list(dataframes.keys())[0]
    print(f"\nPrimary data set for further exploration: {primary_recset_id}")
    print(dataframes[primary_recset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing steps.

- Select a numeric field for analysis (e.g., 'MW [kDa]', 'Normalized Abundance', 'Number of Peptides', etc.).
- Demonstrate filtering, normalization, and grouping.

Fields are referenced by their column names (`@id` in record) as shown above.

In [ ]:
import numpy as np

# Select a numeric field for analysis
df = dataframes[primary_recset_id]
# Try to guess a good numeric column
possible_numeric_fields = [
    'MW [kDa]',                # molecular weight
    'Number of Peptides',      # peptide counts
    'Coverage [%]',            # protein coverage
    'Normalized Abundance',    # normalization
    'Unique',                  # unique peptide count
    'pI',                      # isoelectric point
]
numeric_field = None
for col in df.columns:
    if col in possible_numeric_fields:
        numeric_field = col
        break
# If none found, just use the first numeric-like column
if numeric_field is None:
    numeric_candidates = [col for col in df.columns if df[col].dtype != 'O']
    numeric_field = numeric_candidates[0] if numeric_candidates else df.columns[0]

print(f"Using numeric field for analysis: '{numeric_field}'\n")

# Convert to numeric, coerce errors
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Remove NaNs for this field
filtered_df = df[df[numeric_field].notnull() & (df[numeric_field] > 10)]
print(f"Filtered records with {numeric_field} > 10 (count={len(filtered_df)}):")
print(filtered_df.head())

# Normalize numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()

print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Demonstrate grouping
group_candidates = ['Sample', 'Description', 'Accession']
group_field = None
for col in df.columns:
    if col in group_candidates:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by '{group_field}':")
    print(grouped_df.head())
else:
    print("\nNo suitable group field for grouping in this dataset.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and show relationships (e.g., mean per sample or binned histogram).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# If grouping is possible, show mean per group (barplot)
if group_field and group_field in filtered_df.columns and filtered_df[group_field].nunique() < 30:
    means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values()
    plt.figure(figsize=(10, 5))
    sns.barplot(x=means.index, y=means.values)
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We explored the FAIR² protein abundance dataset, loading the data descriptors and records using `mlcroissant`.

- **Data loading**: The data was ingested from the Croissant schema URL, which pointed to a mass spectrometry dataset on human extracellular vesicles.
- **Metadata**: Authors, citations, and dataset description are automatically extracted.
- **Exploration**: We examined columns, filtered based on a numeric field (e.g., MW [kDa]), applied normalization, and visualized the data.
- **Summary**: This approach can be adapted to other mlcroissant datasets by adjusting record set and field `@id`s.

Refer to [mlcroissant documentation](https://mlcommons-croissant.readthedocs.io/) for more advanced data access and transformation techniques.
